# Exploring DICOM Headers




In [1]:
import pydicom
import os


/home/a.kanamarlapudi001/miniconda3/envs/tf_gpu_env/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## Step 1: Choose a DICOM file to explore

You can specify a path to a DICOM file here, or use the example path below.


In [2]:
dicom_path = "/raid/data01/deephealth/dh_dcm_ast/2.25.319977223756281645905941734320722144570/DXm.2.25.182315515521588145112612132423407523968"

if not os.path.exists(dicom_path):
    print(f"File not found: {dicom_path}")
    print("Please update dicom_path with a valid DICOM file path")
else:
    print(f"Found DICOM file: {os.path.basename(dicom_path)}")


Found DICOM file: DXm.2.25.182315515521588145112612132423407523968


## Step 2: Read DICOM header (read-only)

We use `stop_before_pixels=True` to only read the header, not the pixel data. This is faster and ensures we don't modify anything.


In [3]:
ds = pydicom.dcmread(dicom_path, stop_before_pixels=True)

print(f"Total DICOM elements: {len(ds)}")
print(f"File path: {dicom_path}")


Total DICOM elements: 108
File path: /raid/data01/deephealth/dh_dcm_ast/2.25.319977223756281645905941734320722144570/DXm.2.25.182315515521588145112612132423407523968


## Step 3: Display all DICOM header fields

Here are all the fields available in this DICOM file:


In [4]:
all_fields = []

for elem in ds:
    field_name = elem.name
    tag = str(elem.tag)
    
    try:
        value = elem.value
        if isinstance(value, bytes):
            value_str = f"<bytes: {len(value)} bytes>"
        elif isinstance(value, pydicom.sequence.Sequence):
            value_str = f"<Sequence: {len(value)} items>"
        else:
            value_str = str(value)
            if len(value_str) > 80:
                value_str = value_str[:80] + "..."
    except:
        value_str = str(type(elem.value))
    
    all_fields.append((field_name, tag, value_str))

all_fields.sort(key=lambda x: x[0])

print(f"{'Field Name':<50} {'Tag':<20} {'Value'}")
print("-" * 120)

for field_name, tag, value_str in all_fields:
    print(f"{field_name:<50} {tag:<20} {value_str}")


Field Name                                         Tag                  Value
------------------------------------------------------------------------------------------------------------------------
Accession Number                                   (0008,0050)          DHENZX7UWHI7
Acquisition Device Processing Code                 (0018,1401)          GEMS_FFDM_PV
Anatomic Region Sequence                           (0008,2218)          <Sequence: 1 items>
Anode Target Material                              (0018,1191)          RHODIUM
Bits Allocated                                     (0028,0100)          16
Bits Stored                                        (0028,0101)          12
Body Part Examined                                 (0018,0015)          BREAST
Body Part Thickness                                (0018,11A0)          94
Breast Implant Present                             (0028,1300)          NO
Burned In Annotation                               (0028,0301)          NO
Colli

## Step 4: Useful non-patient-sensitive fields

These fields are commonly useful and typically don't contain patient-identifying information:


In [5]:
useful_fields = [
    "PatientAge", "PatientSex", "PatientBirthDate",
    "StudyDate", "StudyTime", "StudyDescription", "StudyInstanceUID",
    "SeriesInstanceUID", "SeriesNumber", "SeriesDescription",
    "Manufacturer", "ManufacturerModelName", "InstitutionName",
    "Modality", "SOPClassUID", "SOPInstanceUID",
    "ImageLaterality", "ViewPosition",
    "WindowCenter", "WindowWidth", "WindowCenterWidthExplanation",
    "ImagerPixelSpacing", "PixelSpacing", "SliceThickness",
    "Rows", "Columns", "BitsAllocated", "BitsStored",
    "AccessionNumber", "ProtocolName", "ContentDate", "ContentTime"
]

print(f"{'Field Name':<40} {'Available':<15} {'Value'}")
print("-" * 100)

for field_name in useful_fields:
    if hasattr(ds, field_name):
        value = getattr(ds, field_name)
        value_str = str(value)
        if len(value_str) > 60:
            value_str = value_str[:60] + "..."
        print(f"{field_name:<40} {'YES':<15} {value_str}")
    else:
        print(f"{field_name:<40} {'NO':<15} Not found in this file")


Field Name                               Available       Value
----------------------------------------------------------------------------------------------------
PatientAge                               YES             555M
PatientSex                               YES             
PatientBirthDate                         YES             
StudyDate                                YES             
StudyTime                                YES             
StudyDescription                         YES             MAMMO STEREOTACTIC BREAST BIOPSY LEFT W SPECIMEN AND CLIP
StudyInstanceUID                         YES             2.25.319977223756281645905941734320722144570
SeriesInstanceUID                        YES             2.25.160860261163242886366048304383300686596
SeriesNumber                             YES             77370
SeriesDescription                        YES             MAMMO STEREOTACTIC BREAST BIOPSY LEFT W SPECIMEN AND CLIP
Manufacturer                             YES 

## Step 5: Patient-related fields

These fields may contain patient-identifying information and may require approval to use:


In [6]:
patient_fields = [
    "PatientID", "PatientName", "PatientAge", "PatientSex",
    "PatientBirthDate", "PatientRace", "PatientEthnicGroup"
]

print(f"{'Field Name':<40} {'Available':<15} {'Value'}")
print("-" * 100)

for field_name in patient_fields:
    if hasattr(ds, field_name):
        value = getattr(ds, field_name)
        value_str = str(value)
        if len(value_str) > 60:
            value_str = value_str[:60] + "..."
        print(f"{field_name:<40} {'YES':<15} {value_str}")
    else:
        print(f"{field_name:<40} {'NO':<15} Not found in this file")


Field Name                               Available       Value
----------------------------------------------------------------------------------------------------
PatientID                                YES             DHKYULLCXJWR
PatientName                              YES             
PatientAge                               YES             555M
PatientSex                               YES             
PatientBirthDate                         YES             
PatientRace                              NO              Not found in this file
PatientEthnicGroup                       NO              Not found in this file


## Summary

This notebook shows what DICOM header fields are available. All operations are read-only - we never modify the original DICOM files.

To explore a different DICOM file, just update the `dicom_path` variable in Step 1.
